# Generative AI Mini-Project: Single Image Super-Resolution (x4) Challenge
**Master 2 — Specialty I2A**

## Overview and Objectives
Welcome to the Single Image Super-Resolution (SISR) Challenge! In this mini-project, your objective is to design, train, and improve a deep learning model capable of upscaling Low-Resolution (LR) images of size $64 \times 64$ into High-Resolution (HR) images of size $256 \times 256$ (upscaling factor $\times 4$).

This starter notebook provides a complete, working pipeline with a **basic baseline Convolutional Autoencoder**. Your task as a team (4 to 5 students maximum) is to modify and improve this baseline to achieve the highest possible scores on the unseen test set.

### Evaluation Metrics
Your generated test predictions (`submission.mat`) will be evaluated on the online challenge platform using the `sewar` library across three full-reference metrics:
1. **PSNR (Peak Signal-to-Noise Ratio):** Measured in dB (*Higher is better*).
2. **SSIM (Structural Similarity Index):** Ranges from 0 to 1 (*Higher is better*).
3. **RMSE (Root Mean Squared Error):** Pixel-level reconstruction error (*Lower is better*).

## 1. Environment Setup and Dependencies
Run the cell below to install the required libraries (`sewar`, `scipy`, `torch`, `torchvision`, `matplotlib`, `tqdm`, and `pillow`).

In [ ]:
!pip install -q sewar scipy torch torchvision matplotlib tqdm pillow

In [ ]:
import os
import glob
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy.io import savemat

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sewar.full_ref import psnr, ssim, rmse

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation Device: {device}")

## 2. Dataset Extraction and DataLoader Preparation
Make sure you have uploaded `train_data.zip` and `test_lr.zip` to your working directory.
- `train_data.zip` contains paired `LR/` ($64 \times 64$) and `HR/` ($256 \times 256$) PNG patches.
- `test_lr.zip` contains 20 unseen Low-Resolution test images (`001.png` to `020.png`) of size $64 \times 64$.

In [ ]:
# Extract train_data.zip and test_lr.zip automatically if present
if os.path.exists("train_data.zip") and not os.path.exists("train_data"):
    with zipfile.ZipFile("train_data.zip", "r") as zf:
        zf.extractall("train_data")
    print("Successfully extracted 'train_data.zip' into 'train_data/'")

if os.path.exists("test_lr.zip") and not os.path.exists("test_lr"):
    with zipfile.ZipFile("test_lr.zip", "r") as zf:
        zf.extractall("test_lr")
    print("Successfully extracted 'test_lr.zip' into 'test_lr/'")

In [ ]:
class SuperResolutionDataset(Dataset):
    """
    Custom PyTorch Dataset for loading paired LR (64x64) and HR (256x256) PNG images.
    Automatically searches inside root_dir for 'LR' and 'HR' folders.
    """
    def __init__(self, root_dir):
        lr_matches = glob.glob(os.path.join(root_dir, "**", "LR", "*.png"), recursive=True)
        hr_matches = glob.glob(os.path.join(root_dir, "**", "HR", "*.png"), recursive=True)
        
        self.lr_files = sorted(lr_matches)
        self.hr_files = sorted(hr_matches)
        
        if len(self.lr_files) == 0 or len(self.lr_files) != len(self.hr_files):
            raise RuntimeError(
                f"Dataset error: Found {len(self.lr_files)} LR images and {len(self.hr_files)} HR images in '{root_dir}'. "
                "Please verify that 'train_data.zip' is uploaded and extracted."
            )

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        # Load RGB images and normalize pixel values to [0.0, 1.0]
        lr_img = np.array(Image.open(self.lr_files[idx]).convert("RGB"), dtype=np.float32) / 255.0
        hr_img = np.array(Image.open(self.hr_files[idx]).convert("RGB"), dtype=np.float32) / 255.0

        # Convert from H x W x C (NumPy) to C x H x W (PyTorch Tensor)
        lr_tensor = torch.from_numpy(lr_img.transpose(2, 0, 1))
        hr_tensor = torch.from_numpy(hr_img.transpose(2, 0, 1))
        return lr_tensor, hr_tensor

# Instantiate dataset and split into 90% Training and 10% Validation
full_dataset = SuperResolutionDataset("train_data")
val_size = int(0.10 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total Pairs: {len(full_dataset)} | Training Samples: {len(train_dataset)} | Validation Samples: {len(val_dataset)}")

## 3. Baseline Autoencoder Architecture
Below is a deliberately simple Convolutional Autoencoder that encodes $64 \times 64$ images and upsamples them to $256 \times 256$ using two transposed convolution layers.

**How you can improve this baseline:**
- Replace `ConvTranspose2d` with `nn.PixelShuffle(upscale_factor=2)` or bicubic residual learning to eliminate checkerboard artifacts.
- Add **Residual Blocks** (like SRResNet or EDSR) or **Skip Connections** between the encoder and decoder.
- Add **Channel Attention** (RCAN) or increase feature channels (`64` $\rightarrow$ `128`).
- Experiment with **L1 Loss (`nn.L1Loss()`)**, **Charbonnier Loss**, or **VGG Perceptual Loss** instead of standard MSE.

In [ ]:
class DummySRAutoencoder(nn.Module):
    def __init__(self):
        super(DummySRAutoencoder, self).__init__()
        
        # Encoder: Feature extraction in 64x64 Low-Resolution space
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Decoder: Upsampling x4 (64x64 -> 128x128 -> 256x256)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 128x128
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),  # 256x256
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 3, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()  # Constrain outputs to [0.0, 1.0]
        )

    def forward(self, x):
        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)
        return reconstructed

model = DummySRAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")

## 4. Model Training and Loss Curve Visualization
We train the baseline autoencoder for 10 epochs using `MSELoss` and the `Adam` optimizer, tracking both training and validation loss.

In [ ]:
EPOCHS = 10
LEARNING_RATE = 1e-3

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
val_losses = []

for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model.train()
    running_train_loss = 0.0
    for lr_batch, hr_batch in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False):
        lr_batch = lr_batch.to(device)
        hr_batch = hr_batch.to(device)

        optimizer.zero_grad()
        preds = model(lr_batch)
        loss = criterion(preds, hr_batch)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * lr_batch.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    # --- Validation Phase ---
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for lr_batch, hr_batch in val_loader:
            lr_batch = lr_batch.to(device)
            hr_batch = hr_batch.to(device)
            preds = model(lr_batch)
            loss = criterion(preds, hr_batch)
            running_val_loss += loss.item() * lr_batch.size(0)

    epoch_val_loss = running_val_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] -> Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f}")

# --- Plot Training & Validation Loss Curve ---
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o", linewidth=2, label="Training Loss")
plt.plot(range(1, EPOCHS + 1), val_losses, marker="s", linewidth=2, label="Validation Loss")
plt.title("Training and Validation Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.xticks(range(1, EPOCHS + 1))
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Local Validation Evaluation (`sewar`) & Visual Inspection
We compute the three official challenge metrics (**PSNR**, **SSIM**, and **RMSE**) on 20 samples from the validation split using `sewar.full_ref` and display visual reconstructions.

In [ ]:
def tensor_to_uint8_hwc(tensor):
    """
    Converts a PyTorch Tensor of shape (C, H, W) in [0.0, 1.0]
    to a NumPy uint8 array of shape (H, W, C) in [0, 255].
    """
    img_np = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    img_uint8 = np.clip(img_np * 255.0, 0, 255).round().astype(np.uint8)
    return img_uint8

model.eval()
val_psnr, val_ssim, val_rmse = [], [], []
num_eval_samples = min(20, len(val_dataset))

with torch.no_grad():
    for idx in range(num_eval_samples):
        lr_t, hr_t = val_dataset[idx]
        pred_t = model(lr_t.unsqueeze(0).to(device)).squeeze(0)

        gt_uint8 = tensor_to_uint8_hwc(hr_t)
        pred_uint8 = tensor_to_uint8_hwc(pred_t)

        val_psnr.append(float(psnr(gt_uint8, pred_uint8)))
        val_ssim.append(float(ssim(gt_uint8, pred_uint8)[0]))  # ssim returns (ssim_val, cs_val)
        val_rmse.append(float(rmse(gt_uint8, pred_uint8)))

print("=" * 48)
print("   VALIDATION SET BASELINE SCORES (sewar)       ")
print("=" * 48)
print(f"Mean PSNR : {np.mean(val_psnr):.4f} dB  (Higher is better)")
print(f"Mean SSIM : {np.mean(val_ssim):.4f}     (Higher is better)")
print(f"Mean RMSE : {np.mean(val_rmse):.4f}     (Lower is better)")
print("=" * 48)

# --- Visual Comparison on 3 Validation Samples ---
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
with torch.no_grad():
    for row in range(3):
        lr_t, hr_t = val_dataset[row]
        pred_t = model(lr_t.unsqueeze(0).to(device)).squeeze(0)

        lr_vis = tensor_to_uint8_hwc(lr_t)
        pred_vis = tensor_to_uint8_hwc(pred_t)
        hr_vis = tensor_to_uint8_hwc(hr_t)

        axes[row, 0].imshow(lr_vis)
        axes[row, 0].set_title(f"Sample {row+1}: LR Input (64x64)")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(pred_vis)
        axes[row, 1].set_title(f"Predicted HR (PSNR: {psnr(hr_vis, pred_vis):.2f} dB)")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(hr_vis)
        axes[row, 2].set_title(f"Sample {row+1}: Ground Truth HR (256x256)")
        axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

## 6. Generate Test Set Predictions (`submission.mat`)
**Important Submission Instructions:**
- The cell below reads the 20 unseen Low-Resolution PNG images (`001.png` to `020.png`) from `test_lr/`.
- It generates the 20 reconstructed High-Resolution images ($256 \times 256 \times 3$, `uint8`) and packages them into a single compressed MATLAB file named **`submission.mat`** under the key `"predictions"` with exact shape `(20, 256, 256, 3)`.
- Upload **`submission.mat`** directly to the PythonAnywhere challenge website to view your live scores on the leaderboard.

In [ ]:
def find_test_lr_image(index, root_folder="test_lr"):
    """Locates 001.png .. 020.png inside test_lr/."""
    fname = f"{index:03d}.png"
    direct_path = os.path.join(root_folder, fname)
    if os.path.exists(direct_path):
        return direct_path
    matches = glob.glob(os.path.join(root_folder, "**", fname), recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find test image '{fname}' inside '{root_folder}/'.")

model.eval()
predictions_list = []

with torch.no_grad():
    for i in range(1, 21):
        img_path = find_test_lr_image(i, "test_lr")
        
        # Load 64x64 LR PNG image and normalize to [0.0, 1.0]
        lr_img = np.array(Image.open(img_path).convert("RGB"), dtype=np.float32) / 255.0
        lr_tensor = torch.from_numpy(lr_img.transpose(2, 0, 1)).unsqueeze(0).to(device)

        # Generate 256x256 HR prediction
        pred_tensor = model(lr_tensor).squeeze(0)
        pred_uint8 = tensor_to_uint8_hwc(pred_tensor)
        predictions_list.append(pred_uint8)

# Stack into a single 4D uint8 array of shape (20, 256, 256, 3)
predictions_array = np.stack(predictions_list, axis=0)

# Strict verification before saving
assert predictions_array.shape == (20, 256, 256, 3), f"Invalid shape: {predictions_array.shape}. Expected (20, 256, 256, 3)."
assert predictions_array.dtype == np.uint8, f"Invalid dtype: {predictions_array.dtype}. Expected uint8."

# Save only the evaluation file as submission.mat
savemat("submission.mat", {"predictions": predictions_array}, do_compression=True)
size_mb = os.path.getsize("submission.mat") / (1024 * 1024)

print(f"SUCCESS! Generated 'submission.mat' ({size_mb:.2f} MB)")
print(f"Array key: 'predictions' | Shape: {predictions_array.shape} | Dtype: {predictions_array.dtype}")
print("You can now upload 'submission.mat' to the evaluation website!")